In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

In [ ]:
def plot_regularisation(in_vals, R_vals, dF_means=None,
                        x_label=r'$\Delta F_s$', title="", ylim=None,
                        plot_constraints=False):
    """
    Plot multiple regularisation curves R versus ΔF_s in one figure.
    """
    plt.figure(figsize=(6, 4))

    if dF_means is not None:
        for idx, r_vals in enumerate(R_vals):
            ref_val = dF_means[idx]
            plt.plot(in_vals, r_vals,
                     label=fr"$\Delta {{F}}_{{\mathrm{{ref}}}}={ref_val:g}$", linewidth=5)
        plt.legend(frameon=True, fontsize=12)
    else:
        plt.plot(in_vals, R_vals, label='R', linewidth=5)    

    plt.xlabel(x_label, fontsize=18)
    plt.ylabel(r'$R^{\mathrm{diff}}$', fontsize=18)
    plt.title(title)

    plt.xticks(fontsize=14)
    plt.yticks(fontsize=14)

    plt.gca().xaxis.set_major_locator(MultipleLocator(1))  # step of 1 on x-axis

    if ylim is not None:
        plt.ylim(ylim)

    plt.xlim(-1.2, 1.2)
    plt.grid(True)
    plt.show()

In [ ]:
# 1) Define the ranges
num_points = 10000
dF_s_vals = np.linspace(-0.99, 0.99, num_points)

REG DELTA STREAMLINE

In [ ]:
def reg_delta_streamline(dF_s):

    reg = dF_s**2

    return reg

# 3. Compute R for each ΔF_s in that range, using the fixed ΔF_mean
R_vals = reg_delta_streamline(dF_s_vals)
plot_regularisation(dF_s_vals, R_vals, x_label=r'$\Delta F_s$', plot_constraints=True)

REG STREAMLINE + DUAL INV BARRIER

In [ ]:
def reg_delta_streamline(dF_s):

    reg = -1 - 1/((dF_s-1)*(dF_s+1))

    return reg

# 1. Create a range of ΔF_s values
R_vals = reg_delta_streamline(dF_s_vals)
plot_regularisation(dF_s_vals, R_vals, x_label=r'$\Delta F_s$')


REG DELTA REFERENCE

In [ ]:
def reg_delta_fixel(dF_s, dF_mean):
    """
    Vectorized implementation of the piecewise function:
        R(ΔF_s, ΔF_mean) = ((ΔF_s - ΔF_mean)/(1 + ΔF_mean))², if ΔF_s <= ΔF_mean
                           ((ΔF_s - ΔF_mean)/(1 - ΔF_mean))², if ΔF_s > ΔF_mean
    """
    dF_s = np.array(dF_s, dtype=float)  
    dF_mean = np.array(dF_mean, dtype=float)  

    with np.errstate(divide='ignore', invalid='ignore'):
        branch1 = ((dF_s - dF_mean) / (1.0 + dF_mean))**2
        branch2 = ((dF_s - dF_mean) / (1.0 - dF_mean))**2
        R = np.where(dF_s <= dF_mean, branch1, branch2)

    # Replace inf/nan with 1
    R = np.nan_to_num(R, nan=1.0, posinf=1.0, neginf=1.0)

    return R

R_vals = [reg_delta_fixel(dF_s_vals, -0.75), reg_delta_fixel(dF_s_vals, 0), reg_delta_fixel(dF_s_vals, 0.75)]
plot_regularisation(dF_s_vals, R_vals, dF_means=[-0.75, 0, 0.75], x_label=r'$\Delta F_s$', plot_constraints=True)

REG FIXEL/GROUP + DUAL INV BARR

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # For 3D plotting

def reg_dualinvbar_fixel_group(dF_s, dF_mean):
    """
    Vectorized piecewise function for:
      R(ΔF_s, ΔF_mean) = -1 - 1/((X-1)(X+1)),
      where 
        X = (1 + ΔF_s)/(1 + ΔF_mean) - 1     if ΔF_s <= ΔF_mean
        X = (1 + ΔF_s)/(1 - ΔF_mean) + 1     if ΔF_s >  ΔF_mean
    """

    # Ensure NumPy arrays
    dF_s = np.array(dF_s, dtype=float)
    dF_mean = np.array(dF_mean, dtype=float)
    
    # Condition (element-wise)
    cond = (dF_s <= dF_mean)
    
    # X-values for each branch
    X_lower = (1.0 + dF_s) / (1.0 + dF_mean) - 1.0  # ΔF_s <= ΔF_mean
    X_upper = (dF_s - 1.0) / (1.0 - dF_mean) + 1.0  # ΔF_s >  ΔF_mean

    # Corresponding R-values
    R_lower = -1.0 - 1.0 / ((X_lower - 1.0) * (X_lower + 1.0))
    R_upper = -1.0 - 1.0 / ((X_upper - 1.0) * (X_upper + 1.0))

    # Use np.where to combine them
    return np.where(cond, R_lower, R_upper)


R_vals = [reg_dualinvbar_fixel_group(dF_s_vals, -0.75), reg_dualinvbar_fixel_group(dF_s_vals, 0), reg_dualinvbar_fixel_group(dF_s_vals, 0.75)]
plot_regularisation(dF_s_vals, R_vals, dF_means=[-0.75, 0, 0.75], x_label=r'$\Delta F_s$', ylim=(-0.25,5))